In [1]:
!pip install xplique

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 63.9 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ConceptualizingConceptDrift')

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

import torch
import numpy as np

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import random

from experiment_helpers.helper_function import *
from experiment_helpers.driftLocalizer import Localizer
from text_helpers.CraftText import CraftText, CraftTextCombined, full_text_activations
from text_helpers.CustomBertModel import CustomBertForSequenceClassification

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

device = 'cuda'

checkpoint_name = "fabriceyhc/bert-base-uncased-dbpedia_14"
model = CustomBertForSequenceClassification.from_pretrained(checkpoint_name, num_labels=14)
model = model.eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

# dbpedia - non drifting domain
dbpedia = load_dataset("fancyzhx/dbpedia_14")

all_texts = list(dbpedia["train"]["content"])
all_labels = np.array(dbpedia["train"]["label"])

class_ids = [2, 3, 4, 11, 12, 13]

subset_mask = np.isin(all_labels, class_ids)
texts_a = [t for t, m in zip(all_texts, subset_mask) if m]
labels_a = all_labels[subset_mask]

# agnews - introduced gradually (drifting)
ag_news = load_dataset("fancyzhx/ag_news")

texts_b = list(ag_news["train"]["text"])
labels_b = np.array(ag_news["train"]["label"])

print(len(texts_a), len(texts_b))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

...

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

240000 120000


In [5]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
import random

np.random.seed(42)
random.seed(42)

patch_mode = "window"
win_size = 15
stride = 10

drift_localizer = []
localizer_vs_origin = []
c2_domain_a = []
c2_domain_b = []

one_local_l_probs = []
one_local_preds_l_probs = []
one_local_origin_l_probs = []

reconstructed_single_concepts = []
reconstructed_single_concepts_preds = []
reconstructed_all_concepts = []
reconstructed_all_concepts_preds = []

b_ratios = []

run_num = 25
stream_length = 500      
split_point = 0.5        # random split point BD/AD

for j in range(run_num):

    #data stream. share of domain b increases linearly from 0 to 0.8
    t = np.arange(stream_length)
    b_share = 0.8 * (t + 1) / stream_length                    
    b_count = np.floor(np.cumsum(b_share)).astype(int)
    origin = np.diff(b_count, prepend=0)                        

    time_labels = (t >= int(split_point * stream_length)).astype(int)   

    ids_a = np.random.choice(len(texts_a), int((origin == 0).sum()), False)
    ids_b = np.random.choice(len(texts_b), int((origin == 1).sum()), False)

    sample_texts = []
    ia = ib = 0
    for o in origin:
        if o == 0:
            sample_texts.append(texts_a[ids_a[ia]])
            ia += 1
        else:
            sample_texts.append(texts_b[ids_b[ib]])
            ib += 1

    sample_labels = time_labels

    b_ratios.append({"B_total": float(origin.mean()),
                     "B_in_BD": float(origin[time_labels == 0].mean()),
                     "B_in_AD": float(origin[time_labels == 1].mean())})

    patch_act = full_text_activations(sample_texts, model, tokenizer, device=device)
    train_labels = sample_labels

    bd_indices = np.where(sample_labels != 1)[0]
    ad_indices = np.where(sample_labels != 0)[0]

    bd_texts = [sample_texts[i] for i in bd_indices]
    bd_labels = [origin[i] for i in bd_indices]
    ad_texts = [sample_texts[i] for i in ad_indices]
    ad_labels = [origin[i] for i in ad_indices]

    bd_fit = CraftText(model_wrapper=model, tokenizer=tokenizer, num_concepts=10,
                       patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
    print("Fitting Unsupervised Craft....")
    bd_crops, bd_crops_u, bd_w = bd_fit.fit(bd_texts, bd_labels)

    ad_fit = CraftText(model_wrapper=model, tokenizer=tokenizer, num_concepts=10,
                       patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
    print("Fitting Unsupervised Craft....")
    ad_crops, ad_crops_u, ad_w = ad_fit.fit(ad_texts, ad_labels)

    drift_basis = np.vstack([bd_w, ad_w])

    drift_craft = CraftTextCombined(model_wrapper=model, tokenizer=tokenizer, basis=drift_basis,
                                    patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
    print("Fitting Craft....")
    drift_craft.transform_all(sample_texts, origin)      # patch_labels = origin domain

    X_clean = patch_act
    y_clean = train_labels

    localizer_model = Localizer()

    X_train_clean, X_test_clean, y_train, y_test, origin_train, origin_test = \
        train_test_split(X_clean, y_clean, origin, train_size=0.7, random_state=42)

    print('Fitting Random Forest classifier...')
    localizer_model.fit(X_train_clean, y_train)
    print('Fitting complete.')

    # localizer vs time and vs origin domain
    localizer_bin_preds = localizer_model.l_predict(X_test_clean)
    drift_localizer.append(accuracy_score(localizer_bin_preds, y_test))
    localizer_vs_origin.append(accuracy_score(localizer_bin_preds, origin_test))

    proba3 = localizer_model.predict_proba(X_test_clean)     # column2: undecidable therefore non-drifting
    c2_domain_a.append(float(proba3[origin_test == 0, 2].mean()))
    c2_domain_b.append(float(proba3[origin_test == 1, 2].mean()) if (origin_test == 1).any() else np.nan)

    drift_imp = np.round(estimate_importance_l(localizer_model, drift_craft, drift_basis, X_train_clean), 3)

    image_drift_imp_l = [estimate_importance_helper_l(drift_craft, localizer_model, drift_basis,
                                                  image, class_of_interest=localizer_bin_preds[i])
                               for i, image in enumerate(X_test_clean)]

    localizer_bin_train_preds = localizer_model.l_predict(X_train_clean)
    image_drift_imp_l_train = [estimate_importance_helper_l(drift_craft, localizer_model, drift_basis,
                                                  image, class_of_interest=localizer_bin_train_preds[i])
                               for i, image in enumerate(X_train_clean)]
    concept_dist = concept_counter(image_drift_imp_l_train, localizer_bin_train_preds)

    # htilde vs time vs origin domain vs localizer
    one_local_l_probs.append(local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=y_test))
    one_local_preds_l_probs.append(local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=localizer_bin_preds))
    one_local_origin_l_probs.append(local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=origin_test))

    reconstructed_single_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=1)
    localizer_preds = localizer_model.l_predict(reconstructed_single_concept)
    reconstructed_single_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_single_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))

    reconstructed_all_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=len(drift_basis))
    localizer_preds = localizer_model.l_predict(reconstructed_all_concept)
    reconstructed_all_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_all_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))

    print("Run:", j + 1)


Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.36666666666666664 Mean:0.5028571428571429 High threshold:0.6666666666666666, No. Leaves:30
Fitting complete.


Run: 1
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.36666666666666664 Mean:0.5028571428571429 High threshold:0.6666666666666666, No. Leaves:30
Fitting complete.
Run: 2
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.36666666666666664 Mean:0.5028571428571429 High threshold:0.6666666666666666, No. Leaves:30
Fitting complete.
Run: 3
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.38 Mean:0.5028571428571429 High threshold:0.62, No. Leaves:50
Fitting complete.
Run: 4
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Det

In [ ]:
import csv

methods = [drift_localizer,
           localizer_vs_origin,
           c2_domain_a,
           c2_domain_b,

           one_local_l_probs,
           one_local_preds_l_probs,
           one_local_origin_l_probs,

           reconstructed_single_concepts,
           reconstructed_single_concepts_preds,
           reconstructed_all_concepts,
           reconstructed_all_concepts_preds,

           b_ratios]

method_names = ["drift_localizer",
                "localizer_vs_origin",
                "c2_domain_a",
                "c2_domain_b",

                "one_local_l_probs",
                "one_local_preds_l_probs",
                "one_local_origin_l_probs",

                "reconstructed_single_concepts",
                "reconstructed_single_concepts_preds",
                "reconstructed_all_concepts",
                "reconstructed_all_concepts_preds",

                "b_ratios"]

with open('/content/drive/MyDrive/results/text_experiment_gradual_drift.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Method'] + [f'Run_{i+1}' for i in range(run_num)])
    for method, accuracies in zip(method_names, methods):
        writer.writerow([method] + accuracies)


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/results/text_experiment_gradual_drift.csv')
df = df.iloc[:11]

stats = {}
for method in df['Method']:
    accuracies = df[df['Method'] == method].drop('Method', axis=1).values.flatten().astype(float)
    mean = np.nanmean(accuracies)
    std = np.nanstd(accuracies)
    stats[method] = (mean, std)

print(stats)


{'drift_localizer': (np.float64(0.6765333333333333), np.float64(0.024376309081656406)), 'localizer_vs_origin': (np.float64(0.9346666666666665), np.float64(0.04548992562461861)), 'c2_domain_a': (np.float64(0.5698729056543853), np.float64(0.08789450607212063)), 'c2_domain_b': (np.float64(0.3179576921897029), np.float64(0.06946641568605788)), 'one_local_l_probs': (np.float64(0.6717333333333335), np.float64(0.023268863315598385)), 'one_local_preds_l_probs': (np.float64(0.9408000000000001), np.float64(0.03201777284224774)), 'one_local_origin_l_probs': (np.float64(0.9197333333333333), np.float64(0.05182809191076892)), 'reconstructed_single_concepts': (np.float64(0.6658666666666667), np.float64(0.02093970179136063)), 'reconstructed_single_concepts_preds': (np.float64(0.9210666666666667), np.float64(0.0499841752735581)), 'reconstructed_all_concepts': (np.float64(0.6730666666666667), np.float64(0.02410938037823083)), 'reconstructed_all_concepts_preds': (np.float64(0.9426666666666667), np.float6